# Cryptocurrency Price Prediction using Time Series Models

## IBM Machine Learning Assignment

This notebook implements multiple time series forecasting models to predict cryptocurrency prices using historical data from Kaggle.

### Objectives:
1. Explore and preprocess cryptocurrency historical price data
2. Implement three time series models: ARIMA, Prophet, and LSTM
3. Compare model performance and identify the best model
4. Generate insights and business recommendations

### Dataset:
- **Source**: Kaggle - Cryptocurrency Historical Prices Top 100 (2025)
- **Dataset ID**: `isaaclopgu/cryptocurrency-historical-prices-top-100-2025`


In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
from src.utils import set_random_seed
set_random_seed(42)

# Import project modules
from src.data_loader import load_dataset
from src.preprocessing import (
    load_and_clean_data,
    handle_missing_values,
    get_top_cryptocurrencies,
    prepare_time_series_data
)
from src.utils import (
    calculate_metrics,
    split_time_series,
    check_stationarity
)
from src.models.arima_model import (
    fit_arima_model,
    forecast_arima,
    evaluate_arima,
    plot_arima_results,
    plot_acf_pacf
)
from src.models.prophet_model import (
    prepare_prophet_data,
    fit_prophet_model,
    forecast_prophet,
    evaluate_prophet,
    plot_prophet_results,
    plot_prophet_components
)
from src.models.lstm_model import (
    prepare_lstm_data,
    build_lstm_model,
    train_lstm_model,
    predict_lstm,
    evaluate_lstm,
    plot_lstm_training_history,
    plot_lstm_predictions
)
from src.visualization import (
    plot_time_series,
    plot_model_comparison,
    plot_residuals,
    plot_metrics_comparison,
    plot_error_distribution,
    create_metrics_table
)

print("Libraries imported successfully!")


## 1. Data Loading and Exploration


In [ ]:
# Load the dataset
file_path = '../data/crypto_data/Crypto_historical_data.csv'
df_raw = load_dataset(file_path)

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nDate range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")
print(f"\nUnique cryptocurrencies: {df_raw['ticker'].nunique()}")


In [ ]:
# Display first few rows
df_raw.head(10)


In [ ]:
# Get top cryptocurrencies by volume
top_crypto = get_top_cryptocurrencies(df_raw, by='Volume', top_n=10)
print("Top 10 cryptocurrencies by average volume:")
print(top_crypto)


### 1.1 Select Target Cryptocurrency

For this analysis, we'll use **Bitcoin (BTC-USD)** as it has:
- High trading volume
- Long historical data
- Strong market presence


In [ ]:
# Select Bitcoin (BTC-USD) for analysis
target_ticker = 'BTC-USD'
df_btc = load_and_clean_data(file_path, ticker=target_ticker)

print(f"Bitcoin data shape: {df_btc.shape}")
print(f"\nDate range: {df_btc['Date'].min()} to {df_btc['Date'].max()}")
print(f"\nData info:")
df_btc.info()


In [ ]:
# Display summary statistics
df_btc.describe()


## 2. Data Preprocessing


In [ ]:
# Handle missing values
df_btc_clean = handle_missing_values(df_btc, method='forward_fill')
print(f"Missing values after cleaning: {df_btc_clean.isnull().sum().sum()}")
print(f"Shape after cleaning: {df_btc_clean.shape}")

# Prepare time series data
ts_data = prepare_time_series_data(
    df_btc_clean,
    target_col='Close',
    date_col='Date'
)

print(f"\nTime series data shape: {ts_data.shape}")
ts_data.head()


In [ ]:
# Plot time series
plt.figure(figsize=(15, 6))
plt.plot(ts_data['Date'], ts_data['Close'], linewidth=1)
plt.title('Bitcoin (BTC-USD) Price History', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/01_bitcoin_price_history.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Split data into train and test sets (80/20 split)
train_data, test_data = split_time_series(ts_data, date_col='Date', train_ratio=0.8)

print(f"Training data: {train_data.shape[0]} samples ({train_data['Date'].min()} to {train_data['Date'].max()})")
print(f"Test data: {test_data.shape[0]} samples ({test_data['Date'].min()} to {test_data['Date'].max()})")

# Extract time series
train_ts = train_data['Close']
test_ts = test_data['Close']


## 3. ARIMA Model


In [ ]:
# Check stationarity
stationarity_result = check_stationarity(train_ts)
print("Stationarity Test Results:")
print(f"ADF Statistic: {stationarity_result['ADF Statistic']:.4f}")
print(f"p-value: {stationarity_result['p-value']:.6f}")
print(f"Is stationary: {stationarity_result['is_stationary']}")
print(f"\nCritical values:")
for key, value in stationarity_result['Critical Values'].items():
    print(f"  {key}: {value:.4f}")


In [ ]:
# Fit ARIMA model
print("Fitting ARIMA model (this may take a few minutes)...")
arima_result = fit_arima_model(train_ts, auto_select=True)

print(f"\nOptimal ARIMA order: {arima_result['order']}")
print(f"AIC: {arima_result['aic']:.2f}")
print(f"BIC: {arima_result['bic']:.2f}")
print(f"Transformation method: {arima_result['method']}")


In [ ]:
# Generate forecasts for test period
arima_forecast = forecast_arima(
    arima_result['model'],
    steps=len(test_ts),
    timeseries=train_ts,
    method=arima_result['method']
)

# Evaluate on test set
arima_metrics = evaluate_arima(test_ts.values, arima_forecast)
print("ARIMA Model Metrics:")
for metric, value in arima_metrics.items():
    print(f"  {metric}: {value:.4f}")


In [ ]:
# Plot ARIMA results
plt.figure(figsize=(15, 6))
plt.plot(train_data['Date'], train_ts.values, label='Training Data', alpha=0.7)
plt.plot(test_data['Date'], test_ts.values, label='Actual Test', linewidth=2, alpha=0.8)
plt.plot(test_data['Date'], arima_forecast, label='ARIMA Forecast', 
         linestyle='--', linewidth=2, alpha=0.8)
plt.title('ARIMA Model: Predictions vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/02_arima_predictions.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Prophet Model


In [ ]:
# Prepare data for Prophet
prophet_train = prepare_prophet_data(train_data, date_col='Date', target_col='Close')
prophet_test = prepare_prophet_data(test_data, date_col='Date', target_col='Close')

print(f"Prophet training data shape: {prophet_train.shape}")
print(f"Prophet test data shape: {prophet_test.shape}")


In [ ]:
# Fit Prophet model
print("Fitting Prophet model...")
prophet_result = fit_prophet_model(
    prophet_train,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive'
)

print("Prophet model fitted successfully!")
print(f"Parameters: {prophet_result['parameters']}")


In [ ]:
# Generate forecasts for test period
prophet_forecast_df = forecast_prophet(prophet_result['model'], periods=len(test_ts), freq='D')
prophet_forecast = prophet_forecast_df['yhat'].tail(len(test_ts)).values

# Evaluate on test set
prophet_metrics = evaluate_prophet(test_ts.values, prophet_forecast)
print("Prophet Model Metrics:")
for metric, value in prophet_metrics.items():
    print(f"  {metric}: {value:.4f}")


In [ ]:
# Plot Prophet results
plt.figure(figsize=(15, 6))
plt.plot(train_data['Date'], train_ts.values, label='Training Data', alpha=0.7)
plt.plot(test_data['Date'], test_ts.values, label='Actual Test', linewidth=2, alpha=0.8)
plt.plot(test_data['Date'], prophet_forecast, label='Prophet Forecast', 
         linestyle='--', linewidth=2, alpha=0.8)
plt.title('Prophet Model: Predictions vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/03_prophet_predictions.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Plot Prophet components
prophet_result['model'].plot_components(prophet_forecast_df)
plt.savefig('../results/plots/04_prophet_components.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. LSTM Model


In [ ]:
# Prepare data for LSTM
lstm_data = prepare_lstm_data(
    train_ts,
    lookback=60,
    train_ratio=0.9,
    scale=True
)

print(f"LSTM training data shape: {lstm_data['X_train'].shape}")
print(f"LSTM test data shape: {lstm_data['X_test'].shape}")


In [ ]:
# Build LSTM model
lstm_model = build_lstm_model(
    input_shape=(lstm_data['lookback'], 1),
    units=[50, 50],
    dropout_rate=0.2,
    learning_rate=0.001
)

print("LSTM model architecture:")
lstm_model.summary()


In [ ]:
# Train LSTM model
print("Training LSTM model (this may take several minutes)...")
lstm_result = train_lstm_model(
    lstm_model,
    lstm_data['X_train'],
    lstm_data['y_train'],
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    patience=10,
    model_path='../results/models/lstm_model.h5'
)

print("LSTM model training completed!")


In [ ]:
# Plot training history
plot_lstm_training_history(lstm_result['history'], 
                           save_path='../results/plots/05_lstm_training_history.png')
plt.show()


In [ ]:
# Prepare test data for LSTM
# We need to create sequences from the test data
# For simplicity, we'll use the last part of training data to predict test period
# This requires reconstructing sequences from the original time series

# Get the full time series (train + test)
full_ts = ts_data['Close']
full_dates = ts_data['Date']

# Prepare sequences for the test period
# We'll use the last lookback values from train + test data to predict test
test_start_idx = len(train_ts)
lookback = lstm_data['lookback']

# Scale the full time series
scaler = lstm_data['scaler']
full_scaled = scaler.transform(full_ts.values.reshape(-1, 1)).flatten()

# Create sequences for test period
X_test_lstm = []
for i in range(test_start_idx, len(full_scaled)):
    X_test_lstm.append(full_scaled[i-lookback:i])

X_test_lstm = np.array(X_test_lstm).reshape(-1, lookback, 1)

# Make predictions
lstm_forecast_scaled = predict_lstm(lstm_result['model'], X_test_lstm, scaler=None)
lstm_forecast = scaler.inverse_transform(lstm_forecast_scaled.reshape(-1, 1)).flatten()

# Ensure forecast length matches test length
if len(lstm_forecast) > len(test_ts):
    lstm_forecast = lstm_forecast[:len(test_ts)]
elif len(lstm_forecast) < len(test_ts):
    # Pad or truncate as needed
    lstm_forecast = np.pad(lstm_forecast, (0, len(test_ts) - len(lstm_forecast)), mode='edge')


In [ ]:
# Evaluate LSTM model
lstm_metrics = evaluate_lstm(test_ts.values, lstm_forecast)
print("LSTM Model Metrics:")
for metric, value in lstm_metrics.items():
    print(f"  {metric}: {value:.4f}")


In [ ]:
# Plot LSTM results
plt.figure(figsize=(15, 6))
plt.plot(train_data['Date'], train_ts.values, label='Training Data', alpha=0.7)
plt.plot(test_data['Date'], test_ts.values, label='Actual Test', linewidth=2, alpha=0.8)
plt.plot(test_data['Date'], lstm_forecast, label='LSTM Forecast', 
         linestyle='--', linewidth=2, alpha=0.8)
plt.title('LSTM Model: Predictions vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/06_lstm_predictions.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Model Comparison


In [ ]:
# Compile all metrics
all_metrics = {
    'ARIMA': arima_metrics,
    'Prophet': prophet_metrics,
    'LSTM': lstm_metrics
}

# Create metrics table
metrics_df = create_metrics_table(all_metrics, 
                                 save_path='../results/metrics_comparison.csv')
print("Model Comparison Metrics:")
print(metrics_df)


In [ ]:
# Plot metrics comparison
plot_metrics_comparison(all_metrics, 
                       save_path='../results/plots/07_metrics_comparison.png')
plt.show()


In [ ]:
# Plot all predictions together
predictions_dict = {
    'ARIMA': arima_forecast,
    'Prophet': prophet_forecast,
    'LSTM': lstm_forecast
}

plt.figure(figsize=(15, 8))
plt.plot(train_data['Date'], train_ts.values, label='Training Data', 
         alpha=0.6, color='gray')
plt.plot(test_data['Date'], test_ts.values, label='Actual Test', 
         linewidth=2.5, color='black', alpha=0.9)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, (model_name, pred) in enumerate(predictions_dict.items()):
    plt.plot(test_data['Date'], pred, label=f'{model_name} Forecast', 
             linestyle='--', linewidth=2, alpha=0.8, color=colors[i])

plt.title('Model Comparison: All Predictions vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/08_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Plot residuals for each model
for model_name, pred in predictions_dict.items():
    plot_residuals(test_ts.values, pred, model_name=model_name,
                   save_path=f'../results/plots/09_residuals_{model_name.lower()}.png')
    plt.show()


In [ ]:
# Plot error distribution
plot_error_distribution(test_ts.values, predictions_dict,
                        save_path='../results/plots/10_error_distribution.png')
plt.show()


## 7. Key Findings and Insights

### 7.1 Best Model Identification


In [ ]:
# Identify best model for each metric
print("Best Model by Metric:")
print("-" * 40)
for metric in ['MAE', 'RMSE', 'MAPE']:
    best_model = metrics_df[metric].idxmin()
    best_value = metrics_df.loc[best_model, metric]
    print(f"{metric}: {best_model} ({best_value:.4f})")

# Overall best model (lowest average rank)
ranked_metrics = metrics_df.rank(axis=0)
ranked_metrics['Average_Rank'] = ranked_metrics.mean(axis=1)
best_overall = ranked_metrics['Average_Rank'].idxmin()

print(f"\nOverall Best Model: {best_overall}")
print(f"Average Rank: {ranked_metrics.loc[best_overall, 'Average_Rank']:.2f}")


### 7.2 Model Strengths and Weaknesses

**ARIMA:**
- Strengths: Statistical foundation, interpretable, fast training
- Weaknesses: Assumes linear relationships, requires stationarity, may struggle with complex patterns

**Prophet:**
- Strengths: Handles seasonality well, robust to missing data, provides confidence intervals
- Weaknesses: Assumes additive seasonality, less flexible for non-seasonal patterns

**LSTM:**
- Strengths: Can capture complex non-linear patterns, learns long-term dependencies
- Weaknesses: Requires more data, computationally expensive, less interpretable


### 7.3 Business Insights

1. **Volatility**: Cryptocurrency prices exhibit high volatility, making prediction challenging
2. **Trend Detection**: All models capture general trends but struggle with sudden price movements
3. **Model Selection**: The choice of model depends on the specific use case:
   - For short-term trading: LSTM may be preferred
   - For trend analysis: Prophet provides better seasonal insights
   - For quick forecasts: ARIMA is computationally efficient

4. **Risk Management**: High MAPE values indicate significant prediction uncertainty, highlighting the importance of risk management in cryptocurrency investments


## 8. Conclusions

This project successfully implemented three time series forecasting models for cryptocurrency price prediction. Each model has its strengths and is suitable for different applications. The analysis provides valuable insights into cryptocurrency price behavior and demonstrates the challenges and opportunities in financial time series forecasting.
